In [ ]:
# Run first: locate the repository root from any notebook subfolder.
from pathlib import Path
import os
import sys
project_root = next((candidate for candidate in [Path.cwd(), *Path.cwd().parents]
                     if (candidate / "app.py").is_file()
                     and (candidate / "retrieval_final.py").is_file()), None)
if project_root is None:
    raise RuntimeError("Start Jupyter within the project repository.")
os.chdir(project_root)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))


# Hybrid retrieval evaluation

This notebook evaluates all 250 labelled questions against the complete cleaned Enron corpus. It reuses the completed E5 embeddings, builds the TF-IDF vector space once, combines both rankings with reciprocal-rank fusion, and reranks the strongest 20 hybrid candidates with `BAAI/bge-reranker-base`.

The dense search uses the exact FAISS index built by `Embeddings-Final.ipynb`. The saved TF-IDF vectorizer, sparse matrix, and dense index can all be reused for later questions without rebuilding them.

In [ ]:
# Configure the lexical, dense, fusion, and reranking settings used for hybrid retrieval.
import gc
import json
import os
import time
from pathlib import Path

import faiss
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from IPython.display import display
from scipy.sparse import load_npz, save_npz
from sentence_transformers import CrossEncoder, SentenceTransformer
from sklearn.feature_extraction.text import TfidfVectorizer

BASE_DIRECTORY = Path.cwd()
EMAILS_FILE = BASE_DIRECTORY / 'enron_clean.csv'
EVALUATION_FILE = BASE_DIRECTORY / 'data/enron_evaluation_250.csv'
EMBEDDING_DIRECTORY = BASE_DIRECTORY / 'e5_embeddings_final'
MANIFEST_FILE = EMBEDDING_DIRECTORY / 'embedding_manifest.json'
DENSE_INDEX_FILE = EMBEDDING_DIRECTORY / 'e5_dense_index.faiss'

TFIDF_VECTORIZER_FILE = BASE_DIRECTORY / 'full_corpus_tfidf_vectorizer.joblib'
TFIDF_MATRIX_FILE = BASE_DIRECTORY / 'full_corpus_tfidf_matrix.npz'

EMBEDDING_MODEL_NAME = 'intfloat/e5-base-v2'
QUERY_PREFIX = 'query: '
FUSION_DEPTH = 100
DENSE_CHUNK_FETCH = 5_000
RRF_CONSTANT = 60
LEXICAL_WEIGHT = 1.0
DENSE_WEIGHT = 1.0
RERANKER_MODEL_NAME = 'BAAI/bge-reranker-base'
RERANK_DEPTH = 20
RERANKER_BATCH_SIZE = 16
TOP_K_VALUES = [1, 3, 5, 10, 50, 100]

print('Full-corpus hybrid settings are ready.')

In [ ]:
# Load the corpus, questions, and completed embedding manifest, then check that they match.
emails = pd.read_csv(EMAILS_FILE)
evaluation_questions = pd.read_csv(EVALUATION_FILE)
manifest = json.loads(MANIFEST_FILE.read_text(encoding='utf-8'))
if not DENSE_INDEX_FILE.exists():
    raise FileNotFoundError('Run the dense-index cell in Embeddings-Final.ipynb first.')
dense_index = faiss.read_index(str(DENSE_INDEX_FILE))

required_email_columns = {'email_id', 'sender', 'recipients', 'subject', 'body'}
required_question_columns = {'question', 'answer', 'email_id', 'difficulty', 'retrieval_type'}
missing_email_columns = required_email_columns - set(emails.columns)
missing_question_columns = required_question_columns - set(evaluation_questions.columns)
if missing_email_columns or missing_question_columns:
    raise ValueError(
        f'Missing email columns: {sorted(missing_email_columns)}; '
        f'missing question columns: {sorted(missing_question_columns)}'
    )
if not manifest.get('complete'):
    raise RuntimeError('The E5 embedding manifest is not marked complete.')
if manifest['model_name'] != EMBEDDING_MODEL_NAME:
    raise ValueError('The saved embeddings use a different model.')
if manifest['number_of_emails'] != len(emails):
    raise ValueError('The saved embeddings do not match the number of cleaned emails.')
if dense_index.ntotal != manifest['number_of_chunks'] or dense_index.d != manifest['embedding_dimensions']:
    raise ValueError('The FAISS index does not match the completed embedding manifest.')

emails['email_id'] = emails['email_id'].astype(str)
evaluation_questions['email_id'] = evaluation_questions['email_id'].astype(str)
email_ids = emails['email_id'].to_numpy()
email_position_by_id = {email_id: position for position, email_id in enumerate(email_ids)}
missing_sources = set(evaluation_questions['email_id']) - set(email_position_by_id)
if missing_sources:
    raise ValueError(f'{len(missing_sources)} labelled source emails are missing from the full corpus.')
target_positions = np.asarray(
    [email_position_by_id[email_id] for email_id in evaluation_questions['email_id']],
    dtype=np.int64,
)

print(f"Emails: {len(emails):,}")
print(f"Questions: {len(evaluation_questions):,}")
print(f"Saved E5 chunks: {manifest['number_of_chunks']:,}")
print(f"Chunk size / overlap: {manifest['chunk_size']} / {manifest['chunk_overlap']}")

In [ ]:
# Build TF-IDF once, or load it on later runs. This is the lexical vector space.
def build_search_text(email_frame):
    return (
        email_frame['sender'].fillna('') + ' ' +
        email_frame['recipients'].fillna('') + ' ' +
        email_frame['subject'].fillna('') + ' ' +
        email_frame['body'].fillna('')
    )

tfidf_started = time.perf_counter()
if TFIDF_VECTORIZER_FILE.exists() and TFIDF_MATRIX_FILE.exists():
    tfidf_vectorizer = joblib.load(TFIDF_VECTORIZER_FILE)
    email_tfidf_matrix = load_npz(TFIDF_MATRIX_FILE)
    if email_tfidf_matrix.shape[0] != len(emails):
        raise ValueError('The saved TF-IDF matrix does not match enron_clean.csv.')
    print('Loaded the existing full-corpus TF-IDF vector space.')
elif TFIDF_VECTORIZER_FILE.exists() or TFIDF_MATRIX_FILE.exists():
    raise RuntimeError('Only one TF-IDF file exists. Remove the incomplete pair and rerun this cell.')
else:
    tfidf_vectorizer = TfidfVectorizer(
        stop_words='english',
        lowercase=True,
        ngram_range=(1, 2),
        sublinear_tf=True,
        dtype=np.float32,
    )
    email_tfidf_matrix = tfidf_vectorizer.fit_transform(build_search_text(emails))

    temporary_vectorizer = TFIDF_VECTORIZER_FILE.with_suffix('.joblib.tmp')
    temporary_matrix = TFIDF_MATRIX_FILE.with_name(TFIDF_MATRIX_FILE.stem + '.tmp.npz')
    joblib.dump(tfidf_vectorizer, temporary_vectorizer, compress=3)
    save_npz(temporary_matrix, email_tfidf_matrix, compressed=True)
    os.replace(temporary_vectorizer, TFIDF_VECTORIZER_FILE)
    os.replace(temporary_matrix, TFIDF_MATRIX_FILE)
    print('Built and saved the full-corpus TF-IDF vector space.')

print(f'TF-IDF shape: {email_tfidf_matrix.shape}')
print(f'TF-IDF ready in {time.perf_counter() - tfidf_started:.1f} seconds.')

In [ ]:
# Retrieve the top lexical email candidates for each evaluation question using TF-IDF.
def top_k_positive(scores, k):
    positive_positions = np.flatnonzero(scores > 0)
    if len(positive_positions) > k:
        selected = np.argpartition(scores[positive_positions], -k)[-k:]
        positive_positions = positive_positions[selected]
    return positive_positions[np.argsort(scores[positive_positions])[::-1]]

lexical_started = time.perf_counter()
question_tfidf_matrix = tfidf_vectorizer.transform(
    evaluation_questions['question'].astype(str)
)
all_lexical_scores = (question_tfidf_matrix @ email_tfidf_matrix.T).toarray()
lexical_rankings = []
lexical_target_ranks = []
for question_position, scores in enumerate(all_lexical_scores):
    lexical_rankings.append(top_k_positive(scores, FUSION_DEPTH))
    target_score = scores[target_positions[question_position]]
    lexical_target_ranks.append(
        1 + int(np.sum(scores > target_score)) if target_score > 0 else np.inf
    )
lexical_target_ranks = np.asarray(lexical_target_ranks, dtype=float)
del all_lexical_scores
print(f'Ranked all questions with TF-IDF in {time.perf_counter() - lexical_started:.1f} seconds.')

In [ ]:
# Encode all evaluation questions as normalized E5 query embeddings.
if torch.cuda.is_available():
    embedding_device = 'cuda'
elif torch.backends.mps.is_available():
    embedding_device = 'mps'
else:
    embedding_device = 'cpu'

embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME, device=embedding_device)
question_embeddings = embedding_model.encode(
    [QUERY_PREFIX + str(question) for question in evaluation_questions['question']],
    batch_size=64,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True,
).astype(np.float32)
if question_embeddings.shape[1] != manifest['embedding_dimensions']:
    raise ValueError('Question and saved passage embedding dimensions do not match.')
print(f'Question embeddings created on {embedding_device}.')

In [ ]:
# Retrieve the top dense email candidates and build the chunk-to-email lookup needed for reranking.
dense_started = time.perf_counter()
number_of_questions = len(evaluation_questions)
number_of_emails = len(emails)
email_shard = np.full(number_of_emails, -1, dtype=np.int32)
email_local_start = np.full(number_of_emails, -1, dtype=np.int32)
email_local_end = np.full(number_of_emails, -1, dtype=np.int32)
chunk_source_rows = np.empty(dense_index.ntotal, dtype=np.int32)
shard_global_start_by_number = {}
global_offset = 0

for shard in manifest['shards']:
    shard_global_start_by_number[int(shard['shard_number'])] = global_offset
    chunks_path = BASE_DIRECTORY / shard['chunks_file']
    source_rows = pd.read_parquet(chunks_path, columns=['source_row'])['source_row'].to_numpy(dtype=np.int64)
    if len(source_rows) == 0:
        continue
    next_offset = global_offset + len(source_rows)
    chunk_source_rows[global_offset:next_offset] = source_rows
    local_starts = np.r_[0, np.flatnonzero(source_rows[1:] != source_rows[:-1]) + 1]
    local_ends = np.r_[local_starts[1:], len(source_rows)]
    shard_email_positions = source_rows[local_starts]
    email_shard[shard_email_positions] = int(shard['shard_number'])
    email_local_start[shard_email_positions] = local_starts
    email_local_end[shard_email_positions] = local_ends
    global_offset = next_offset

if global_offset != dense_index.ntotal:
    raise ValueError('The vector-to-email lookup does not match the FAISS index.')
chunk_fetch = min(DENSE_CHUNK_FETCH, dense_index.ntotal)
chunk_scores, chunk_vector_ids = dense_index.search(question_embeddings, chunk_fetch)
dense_rankings = []
for vector_ids in chunk_vector_ids:
    seen = set()
    ranking = []
    for vector_id in vector_ids:
        email_position = int(chunk_source_rows[vector_id])
        if email_position not in seen:
            seen.add(email_position)
            ranking.append(email_position)
            if len(ranking) == FUSION_DEPTH:
                break
    if len(ranking) < FUSION_DEPTH:
        raise RuntimeError('Increase DENSE_CHUNK_FETCH to obtain 100 unique email candidates.')
    dense_rankings.append(np.asarray(ranking, dtype=np.int64))
dense_target_ranks = []
for ranking, target_position in zip(dense_rankings, target_positions):
    matches = np.flatnonzero(ranking == target_position)
    dense_target_ranks.append(int(matches[0]) + 1 if len(matches) else np.inf)
dense_target_ranks = np.asarray(dense_target_ranks, dtype=float)
print(f'FAISS E5 search finished in {time.perf_counter() - dense_started:.1f} seconds.')

In [ ]:
# Fuse the TF-IDF and E5 rankings using reciprocal rank fusion.
def reciprocal_rank_fusion(lexical_ranking, dense_ranking):
    scores = {}
    for rank, email_position in enumerate(lexical_ranking[:FUSION_DEPTH], start=1):
        scores[int(email_position)] = scores.get(int(email_position), 0.0) + (
            LEXICAL_WEIGHT / (RRF_CONSTANT + rank)
        )
    for rank, email_position in enumerate(dense_ranking[:FUSION_DEPTH], start=1):
        scores[int(email_position)] = scores.get(int(email_position), 0.0) + (
            DENSE_WEIGHT / (RRF_CONSTANT + rank)
        )
    return np.asarray(
        sorted(scores, key=lambda position: scores[position], reverse=True),
        dtype=np.int64,
    )

hybrid_rankings = [
    reciprocal_rank_fusion(lexical_rankings[i], dense_rankings[i])
    for i in range(number_of_questions)
]

def ranks_from_candidate_lists(candidate_lists):
    ranks = []
    for candidates, target_position in zip(candidate_lists, target_positions):
        matches = np.flatnonzero(candidates == target_position)
        ranks.append(int(matches[0]) + 1 if len(matches) else np.inf)
    return np.asarray(ranks, dtype=float)

hybrid_target_ranks = ranks_from_candidate_lists(hybrid_rankings)
print(f'Created {number_of_questions} hybrid rankings.')

In [ ]:
# Compare TF-IDF, E5, and hybrid retrieval performance before reranking.
def metric_record(method, ranks):
    row = {'method': method}
    for k in TOP_K_VALUES:
        row[f'recall_at_{k}'] = float(np.mean(ranks <= k))
    row['mrr'] = float(np.mean(1.0 / ranks))
    return row

first_stage_ranks = {
    'TF-IDF': lexical_target_ranks,
    'Dense E5': dense_target_ranks,
    'Hybrid RRF': hybrid_target_ranks,
}
first_stage_results = pd.DataFrame([
    metric_record(method, ranks) for method, ranks in first_stage_ranks.items()
]).sort_values(['recall_at_10', 'mrr'], ascending=False).reset_index(drop=True)
display(first_stage_results.style.format({
    **{f'recall_at_{k}': '{:.1%}' for k in TOP_K_VALUES},
    'mrr': '{:.3f}',
}).background_gradient(subset=[f'recall_at_{k}' for k in TOP_K_VALUES] + ['mrr'], cmap='YlGn'))

In [ ]:
# Select the strongest chunk from each top hybrid email as focused evidence for the reranker.
candidate_records = []
for question_position, ranking in enumerate(hybrid_rankings):
    for hybrid_rank, email_position in enumerate(ranking[:RERANK_DEPTH], start=1):
        candidate_records.append({
            'question_position': question_position,
            'question': str(evaluation_questions.iloc[question_position]['question']),
            'email_position': int(email_position),
            'email_id': email_ids[email_position],
            'hybrid_rank': hybrid_rank,
            'shard_number': int(email_shard[email_position]),
            'local_start': int(email_local_start[email_position]),
            'local_end': int(email_local_end[email_position]),
        })
reranker_candidates = pd.DataFrame(candidate_records)
reranker_candidates['chunk_number'] = -1
reranker_candidates['passage'] = ''

shards_by_number = {int(shard['shard_number']): shard for shard in manifest['shards']}
for shard_number, group in reranker_candidates.groupby('shard_number'):
    if shard_number < 0:
        continue
    shard = shards_by_number[int(shard_number)]
    shard_chunks = pd.read_parquet(
        BASE_DIRECTORY / shard['chunks_file'],
        columns=['chunk_number', 'chunk_text'],
    )
    for row_index, candidate in group.iterrows():
        start = int(candidate['local_start'])
        end = int(candidate['local_end'])
        question_position = int(candidate['question_position'])
        email_position = int(candidate['email_position'])
        global_start = shard_global_start_by_number[int(shard_number)] + start
        candidate_vectors = dense_index.reconstruct_n(global_start, end - start)
        local_scores = candidate_vectors @ question_embeddings[question_position]
        best_local_row = start + int(np.argmax(local_scores))
        chunk = shard_chunks.iloc[best_local_row]
        subject = '' if pd.isna(emails.iloc[email_position]['subject']) else str(emails.iloc[email_position]['subject'])
        reranker_candidates.at[row_index, 'chunk_number'] = int(chunk['chunk_number'])
        reranker_candidates.at[row_index, 'passage'] = (
            f"Subject: {subject}\n{str(chunk['chunk_text'])}"
        )

if (reranker_candidates['passage'].str.len() == 0).any():
    raise RuntimeError('At least one reranker passage could not be prepared.')
print(f'Prepared {len(reranker_candidates):,} question-passage pairs.')

In [ ]:
# Run the BGE cross-encoder locally and replace only the first 20 hybrid positions.
if torch.cuda.is_available():
    reranker_device = 'cuda'
elif torch.backends.mps.is_available():
    reranker_device = 'mps'
else:
    reranker_device = 'cpu'

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
if torch.backends.mps.is_available():
    torch.mps.empty_cache()

reranker_started = time.perf_counter()
reranker = CrossEncoder(RERANKER_MODEL_NAME, max_length=512, device=reranker_device)
pairs = list(zip(reranker_candidates['question'], reranker_candidates['passage']))
reranker_candidates['reranker_score'] = np.asarray(
    reranker.predict(
        pairs,
        batch_size=RERANKER_BATCH_SIZE,
        show_progress_bar=True,
    )
).reshape(-1)

reranked_rankings = []
for question_position, original_ranking in enumerate(hybrid_rankings):
    reranked_prefix = (
        reranker_candidates[reranker_candidates['question_position'] == question_position]
        .sort_values(['reranker_score', 'hybrid_rank'], ascending=[False, True])['email_position']
        .to_numpy(dtype=np.int64)
    )
    reranked_rankings.append(np.concatenate([reranked_prefix, original_ranking[len(reranked_prefix):]]))
reranker_target_ranks = ranks_from_candidate_lists(reranked_rankings)
print(
    f'Reranked {len(reranker_candidates):,} pairs on {reranker_device} '
    f'in {time.perf_counter() - reranker_started:.1f} seconds.'
)

In [ ]:
# Compare retrieval performance across TF-IDF, E5, hybrid fusion, and reranking.
all_method_ranks = {
    **first_stage_ranks,
    f'Hybrid RRF + BGE reranker (top {RERANK_DEPTH})': reranker_target_ranks,
}
final_results = pd.DataFrame([
    metric_record(method, ranks) for method, ranks in all_method_ranks.items()
]).sort_values(['recall_at_10', 'mrr'], ascending=False).reset_index(drop=True)
display(final_results.style.format({
    **{f'recall_at_{k}': '{:.1%}' for k in TOP_K_VALUES},
    'mrr': '{:.3f}',
}).background_gradient(subset=[f'recall_at_{k}' for k in TOP_K_VALUES] + ['mrr'], cmap='YlGn'))

plot_data = final_results.melt(
    id_vars='method',
    value_vars=['recall_at_1', 'recall_at_5', 'recall_at_10', 'recall_at_50', 'recall_at_100'],
    var_name='metric',
    value_name='score',
)
plt.figure(figsize=(12, 5))
sns.barplot(data=plot_data, x='metric', y='score', hue='method')
plt.ylim(0, 1)
plt.xlabel('Metric')
plt.ylabel('Score')
plt.title('Full-corpus retrieval performance on 250 grounded questions')
plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.show()

In [ ]:
# Compare TF-IDF, E5, hybrid, and reranked accuracy for easy, medium, and hard questions.
difficulty_rows = []
for difficulty, group in evaluation_questions.groupby('difficulty', sort=False):
    positions = group.index.to_numpy(dtype=np.int64)
    for method, ranks in all_method_ranks.items():
        row = metric_record(method, ranks[positions])
        row['difficulty'] = difficulty
        row['questions'] = len(positions)
        difficulty_rows.append(row)
difficulty_results = pd.DataFrame(difficulty_rows)[
    ['difficulty', 'method', 'questions'] + [f'recall_at_{k}' for k in TOP_K_VALUES] + ['mrr']
]
display(difficulty_results.style.format({
    **{f'recall_at_{k}': '{:.1%}' for k in TOP_K_VALUES}, 'mrr': '{:.3f}'
}).background_gradient(subset=[f'recall_at_{k}' for k in TOP_K_VALUES] + ['mrr'], cmap='YlGn'))

plt.figure(figsize=(13, 6))
sns.barplot(data=difficulty_results, x='difficulty', y='recall_at_10', hue='method', order=['easy', 'medium', 'hard'])
plt.ylim(0, 1)
plt.xlabel('Question difficulty')
plt.ylabel('Recall@10')
plt.title('Full-corpus Recall@10 by difficulty and retrieval stage')
plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.show()